In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")


In [ ]:
LANDMARK_MAP_EN = {
    "lm11": "left_shoulder",   # ombro esquerdo
    "lm12": "right_shoulder",  # ombro direito
    "lm13": "left_elbow",      # cotovelo esquerdo
    "lm14": "right_elbow",     # cotovelo direito
    "lm15": "left_wrist",      # punho esquerdo
    "lm16": "right_wrist",     # punho direito
    "lm23": "left_hip",        # quadril esquerdo
    "lm24": "right_hip",       # quadril direito
}

LANDMARK_MAP = {
    "lm11": "ombro_esquerdo",   # ombro esquerdo
    "lm12": "ombro_direito",  # ombro direito
    "lm13": "cotovelo_esquerdo",      # cotovelo esquerdo
    "lm14": "cotovelo_direito",     # cotovelo direito
    "lm15": "punho_esquerdo",      # punho esquerdo
    "lm16": "punho_direito",     # punho direito
    "lm23": "quadril_esquerdo",        # quadril esquerdo
    "lm24": "quadril_direito",       # quadril direito
}

In [ ]:
csv_path = r"./saida/landmarks/my_record_02_landmarks.csv" 
df = pd.read_csv(csv_path)

# renomeia lmXX_{x,y,z,vis} -> {nome_anatomico}_{x,y,z,vis}
rename = {}
for lm, name in LANDMARK_MAP.items():
    for suf in ("x", "y", "z", "vis"):
        old = f"{lm}_{suf}"
        new = f"{name}_{suf}"
        if old in df.columns:
            rename[old] = new

df = df.rename(columns=rename)
df.head()


In [ ]:
id_vars = [c for c in ["frame", "time_sec"] if c in df.columns]
if "time_sec" not in df.columns:
    raise ValueError("A coluna 'time_sec' não existe no CSV.")

coord_cols = []
for name in LANDMARK_MAP.values():
    for c in ("x", "y", "z"):
        col = f"{name}_{c}"
        if col in df.columns:
            coord_cols.append(col)

if not coord_cols:
    raise ValueError("Não encontrei colunas de coordenadas após renomear. Confira os nomes do CSV.")

long = df.melt(
    id_vars=id_vars,
    value_vars=coord_cols,
    var_name="feature",
    value_name="value"
)

# feature: "left_shoulder_x" -> landmark="left_shoulder", coord="x"
long[["landmark", "coord"]] = long["feature"].str.rsplit("_", n=1, expand=True)
long = long.drop(columns=["feature"])

# visibilidade (se existir)
vis_cols = [f"{name}_vis" for name in LANDMARK_MAP.values() if f"{name}_vis" in df.columns]
if vis_cols:
    vis_long = df.melt(
        id_vars=id_vars,
        value_vars=vis_cols,
        var_name="vis_feature",
        value_name="vis"
    )
    vis_long["landmark"] = vis_long["vis_feature"].str.replace("_vis", "", regex=False)
    vis_long = vis_long.drop(columns=["vis_feature"])
    long = long.merge(vis_long, on=id_vars + ["landmark"], how="left")
else:
    long["vis"] = None

long.head()


In [ ]:
min_vis = 0.5  # ajuste ou coloque None pra não filtrar

plot_data = long.copy()
if min_vis is not None:
    plot_data = plot_data[(plot_data["vis"].isna()) | (plot_data["vis"] >= min_vis)]


In [ ]:
g = sns.relplot(
    data=plot_data,
    x="time_sec",
    y="value",
    kind="line",
    row="landmark",
    col="coord",
    height=2.2,
    aspect=1.6,
    facet_kws=dict(sharex=True, sharey=False)
)

g.set_axis_labels("Tempo (s)", "Coordenada (normalizada)")
g.set_titles(col_template="{col_name}")
for ax, col in zip(g.axes[0], g.col_names):
    if col == "x":
        ax.set_title(col, fontsize=16)
    else:
        ax.set_title(col, fontsize=9)

#g.fig.suptitle("Pontos articulares - séries temporais", y=1.02)
plt.show()


In [ ]:
# pega só coord x e y
xy = plot_data[plot_data["coord"].isin(["x", "y"])].copy()

# pivot: uma linha por (frame/time/landmark) com colunas x e y
xy_piv = xy.pivot_table(index=id_vars + ["landmark"], columns="coord", values="value").reset_index()

# junta vis (se tiver)
if "vis" in plot_data.columns and plot_data["vis"].notna().any():
    vis_one = plot_data.dropna(subset=["vis"]).drop_duplicates(subset=id_vars + ["landmark"])[id_vars + ["landmark", "vis"]]
    xy_piv = xy_piv.merge(vis_one, on=id_vars + ["landmark"], how="left")

g2 = sns.relplot(
    data=xy_piv,
    x="x",
    y="y",
    hue="time_sec",
    row="landmark",
    kind="scatter",
    height=2.2,
    aspect=1.6,
    #col_wrap=4,
    palette="viridis",
    s=10,
    linewidth=0
)
g2.set_xlabels("X", fontsize=15)
g2.set_ylabels("Y")
g2.fig.suptitle("Pose Landmarks — trajetória XY (cor = tempo)", y=1.02)
plt.show()


In [ ]:
import pandas as pd
import cv2
import os

def gerar_miniaturas(csv_path, video_path, output_dir):
    # Carregar o CSV
    df = pd.read_csv(csv_path)
    
    # Garantir que o diretório de saída exista
    os.makedirs(output_dir, exist_ok=True)
    
    # Abrir o vídeo
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Erro ao abrir o vídeo: {video_path}")
        return
    
    # Processar cada repetição no CSV
    for index, row in df.iterrows():
        repeticao = row['repo_id']  # Coluna com o número da repetição
        frame_id = row['frame']      # Coluna com o número do frame
        
        # Ir para o frame especificado
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            print(f"Erro ao capturar o frame {frame_id} para a repetição {repeticao}")
            continue
        
        # Salvar o frame como imagem
        output_path = os.path.join(output_dir, f"repeticao_{repeticao}.jpg")
        cv2.imwrite(output_path, frame)
        print(f"Miniatura salva: {output_path}")
    
    # Liberar o vídeo
    cap.release()
    print("Miniaturas geradas com sucesso!")

# Exemplo de uso
csv_path = r"./saida/reps_detectadas/me_barbell_25_lateral_reps.csv"
video_path = r"./saida/videos/me_barbell_25_lateral_pose.mp4"
output_dir = r"saida/miniaturas"

gerar_miniaturas(csv_path, video_path, output_dir)

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np

def gerar_mosaico(csv_path, video_path, output_path, mosaico_dimensao=(3, 3), altura_thumb=300):
    """
    Gera um mosaico de frames do vídeo baseado em um CSV.
    
    Args:
        csv_path: caminho do CSV com informações das repetições
        video_path: caminho do vídeo
        output_path: caminho onde salvar o mosaico
        mosaico_dimensao: tupla (linhas, colunas) do mosaico
        altura_thumb: altura desejada para cada thumbnail (mantém aspect ratio)
    """
    # Verificar se o arquivo de vídeo existe
    if not os.path.exists(video_path):
        print(f"Arquivo de vídeo não encontrado: {video_path}")
        return
    
    # Carregar o CSV
    if not os.path.exists(csv_path):
        print(f"Arquivo CSV não encontrado: {csv_path}")
        return
    
    df = pd.read_csv(csv_path)
    
    # Verificar se a coluna 'peak_frame' existe
    if 'peak_frame' not in df.columns:
        print(f"Colunas disponíveis no CSV: {df.columns}")
        raise ValueError("A coluna 'peak_frame' não existe no CSV. Verifique o arquivo.")
    
    # Abrir o vídeo
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Erro ao abrir o vídeo: {video_path}")
        return
    
    # Lista para armazenar os frames capturados
    frames = []
    
    # Processar cada repetição no CSV
    for index, row in df.iterrows():
        frame_id = int(row['peak_frame'])
        
        # Ir para o frame especificado
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        if not ret:
            print(f"Erro ao capturar o frame {frame_id}")
            continue
        
        # Redimensionar mantendo aspect ratio
        h, w = frame.shape[:2]
        scale = altura_thumb / h
        new_w = int(w * scale)
        frame_resized = cv2.resize(frame, (new_w, altura_thumb))
        frames.append(frame_resized)
    
    # Liberar o vídeo
    cap.release()
    
    if not frames:
        print("Nenhum frame foi capturado!")
        return
    
    # Calcular o número de linhas e colunas do mosaico
    num_frames = len(frames)
    rows, cols = mosaico_dimensao
    
    # Garantir que o mosaico tenha espaço suficiente para todos os frames
    if num_frames > rows * cols:
        print(f"Aviso: {num_frames} frames capturados, mas só cabem {rows * cols} no mosaico. Usando apenas os primeiros.")
        frames = frames[:rows * cols]
    
    # Preencher frames faltantes com imagens em branco (se houver)
    while len(frames) < rows * cols:
        frames.append(np.zeros_like(frames[0]))
    
    # Calcular dimensões do mosaico (todas as imagens têm a mesma altura)
    altura = altura_thumb
    
    # Calcular a largura total do mosaico (máximo entre as larguras)
    max_largura = max(f.shape[1] for f in frames[:cols])
    
    # Criar uma imagem para o mosaico
    altura_total = altura * rows
    largura_total = max_largura * cols
    mosaico = np.zeros((altura_total, largura_total, 3), dtype=np.uint8)
    
    # Preencher o mosaico com os frames
    for idx, frame in enumerate(frames):
        r = idx // cols
        c = idx % cols
        
        # Posicionar o frame no mosaico (centralizado ou alinhado à esquerda)
        y_start = r * altura
        x_start = c * max_largura
        frame_h, frame_w = frame.shape[:2]
        
        mosaico[y_start:y_start + frame_h, x_start:x_start + frame_w] = frame
    
    # Salvar o mosaico como uma única imagem
    cv2.imwrite(output_path, mosaico)
    print(f"Mosaico salvo em: {output_path}")
    print(f"Dimensões do mosaico: {mosaico.shape[1]}x{mosaico.shape[0]} pixels")


In [ ]:
# Exemplo de uso
csv_path = r"./saida/reps_detectadas/me_barbell_25_lateral_reps.csv"
video_path = r"./entrada/me_barbell_25_lateral.mov"
output_path = r"./saida/miniaturas/mosaico.jpg"

# altura_thumb: altura em pixels para cada thumbnail (mantém aspect ratio)
# mosaico_dimensao: (linhas, colunas) do mosaico
gerar_mosaico(csv_path, video_path, output_path, mosaico_dimensao=(2, 5), altura_thumb=400)


# Graphs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Path to the source CSV file and output folder for figures
csv_path = Path("saida/resultados/cv_results.csv")
out_dir = Path("saida/resultados")

# Load cross-validation results
df = pd.read_csv(csv_path)

# Rename columns to English
metric_map = {
    'acc': 'Accuracy',
    'f1': 'F1 Score',
    'precision': 'Precision',
    'recall': 'Recall',
    'roc_auc': 'ROC AUC',
    'pr_auc': 'PR AUC'
}
# metric_map = {
#     'acc': 'Acurácia',
#     'f1': 'Score F1',
#     'precision': 'Precisão',
#     'recall': 'Revocação',
#     'roc_auc': 'ROC AUC',
#     'pr_auc': 'PR AUC'
# }
df = df.rename(columns=metric_map)

# Define metrics of interest (excluding 'fold' and threshold column 'thr')
metrics = list(metric_map.values())

# Set a publication-friendly visual style
sns.set_theme(style="whitegrid")

# ==========================================
# Plot 1: Performance by Fold (Bar Plot)
# ==========================================
plt.figure(figsize=(12, 6))

# Melt the DataFrame to long format for Seaborn
df_melted = df.melt(id_vars='fold', value_vars=metrics, var_name='Metric', value_name='Value')

# Create bar plot with publication-friendly palette
ax = sns.barplot(data=df_melted, x='fold', y='Value', hue='Metric', palette='viridis')

plt.title('Metric Performance by Fold (Cross-Validation)', fontsize=14, weight='bold')
plt.xlabel('Fold', fontsize=12, weight='bold')
plt.ylabel('Score (0 to 1)', fontsize=12, weight='bold')

# Move legend outside to avoid overlap
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Metrics')
plt.ylim(0.5, 1.05)  # Start Y axis at 0.5 to emphasize differences at high scores

plt.tight_layout()
plt.savefig(out_dir / 'cv_metrics_per_fold.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# Plot 2: Overall Distribution (Boxplot + Stripplot)
# ==========================================
plt.figure(figsize=(10, 6))

# Boxplot shows median and quartiles
sns.boxplot(data=df[metrics], palette='pastel', width=0.5)
# Stripplot adds individual points to show real dispersion
sns.stripplot(data=df[metrics], color=".25", size=10, jitter=True, alpha=0.7)

plt.ylabel('Score (0 to 1)', fontsize=15, weight='bold')
plt.xlabel('Metric', fontsize=15, weight='bold')
plt.ylim(0.6, 1.02)  # Visual adjustment based on data

plt.tight_layout()
plt.savefig(out_dir / 'cv_metrics_distribution.png', dpi=400)
plt.show()

# ==========================================
# Statistical Summary in Text
# ==========================================
print("=== Summary Statistics (Mean +/- Standard Deviation) ===")
stats_summary = df[metrics].agg(['mean', 'std']).T
for idx, row in stats_summary.iterrows():
    print(f"{idx.ljust(10)}: {row['mean']:.4f} +/- {row['std']:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="paper")

pred_path = Path("saida/resultados/cv_predictions.csv")
preds = pd.read_csv(pred_path)

required_cols = {"fold", "video_id", "rep_id", "y_true", "y_pred"}
missing = required_cols - set(preds.columns)
if missing:
    raise ValueError(f"Colunas ausentes em {pred_path}: {sorted(missing)}")

# Tipagem e ordenacao para garantir a normalizacao temporal correta
preds["fold"] = pd.to_numeric(preds["fold"], errors="coerce").astype("Int64")
preds["rep_id"] = pd.to_numeric(preds["rep_id"], errors="coerce")
preds["y_true"] = pd.to_numeric(preds["y_true"], errors="coerce").fillna(0).astype(int)
preds["y_pred"] = pd.to_numeric(preds["y_pred"], errors="coerce").fillna(0).astype(int)

preds = preds.dropna(subset=["fold", "video_id", "rep_id"]).copy()
preds = preds.sort_values(["fold", "video_id", "rep_id"]).reset_index(drop=True)

# Normaliza rep_id por sequencia (fold + video_id) para [0,1]
def normalize_rep_id(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s), dtype=float), index=s.index)
    return (s - lo) / (hi - lo)

preds["rep_norm"] = preds.groupby(["fold", "video_id"], group_keys=False)["rep_id"].apply(normalize_rep_id)
preds["status"] = np.where(preds["y_true"] == preds["y_pred"], "acerto", "erro")
preds["correct"] = (preds["status"] == "acerto").astype(int)

# Pequeno jitter vertical para reduzir sobreposicao de pontos em y=0 e y=1
rng = np.random.default_rng(2609)
preds["y_pred_jitter"] = preds["y_pred"] + rng.uniform(-0.035, 0.035, size=len(preds))

# Cores e marcadores acessiveis (colorblind-safe + distinguiveis em P&B)
palette = {"acerto": "#0072B2", "erro": "#D55E00"}  # Okabe-Ito
markers = {"acerto": "o", "erro": "X"}

# =============================
# Grafico 1: pontos (acerto/erro)
# =============================
fig1, ax1 = plt.subplots(figsize=(12, 5.8))

sns.scatterplot(
    data=preds,
    x="rep_norm",
    y="y_pred_jitter",
    hue="status",
    style="status",
    markers=markers,
    hue_order=["acerto", "erro"],
    style_order=["acerto", "erro"],
    palette=palette,
    s=28,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.25,
    ax=ax1,
)

#ax1.set_title("Predições por tempo normalizado (pontos de acerto/erro)")
ax1.set_xlabel("Tempo normalizado da repetição", fontsize=15, weight="bold")
ax1.set_ylabel("Classe predita", fontsize=15, weight="bold")
ax1.set_ylim(-0.1, 1.1)
ax1.set_yticks([0, 1])
ax1.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=2,
    frameon=True,
    fontsize=17,
)

fig1.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

# =======================================
# Grafico 2: linha de % acerto acumulado
# =======================================
n_bins = 40
cum_df = preds.sort_values("rep_norm").copy()
cum_df["bin"] = pd.cut(
    cum_df["rep_norm"],
    bins=np.linspace(0, 1, n_bins + 1),
    include_lowest=True,
    labels=False,
)

bin_stats = (
    cum_df.groupby("bin", as_index=False)
    .agg(
        rep_norm=("rep_norm", "mean"),
        n=("correct", "size"),
        acertos=("correct", "sum"),
    )
    .sort_values("rep_norm")
)

bin_stats["cum_acc"] = bin_stats["acertos"].cumsum() / bin_stats["n"].cumsum()

fig2, ax2 = plt.subplots(figsize=(12, 4.6))
ax2.plot(
    bin_stats["rep_norm"],
    100 * bin_stats["cum_acc"],
    color="black",
    linewidth=2.0,
)
#ax2.set_title("Percentual de acerto acumulado ao longo do tempo normalizado")
ax2.set_xlabel("tempo normalizado da repetição", fontsize=15, weight="bold")
ax2.set_ylabel("% acerto acumulado", fontsize=15, weight="bold")
ax2.set_ylim(80, 110)
ax2.set_yticks(np.arange(80, 111, 5))
ax2.grid(True, which="major", axis="both", alpha=0.35)

plt.tight_layout()
plt.show()

summary = (
    preds.groupby(["fold", "video_id"], as_index=False)
    .agg(
        n_reps=("rep_id", "size"),
        acc=("correct", "mean"),
    )
    .sort_values(["fold", "video_id"])
)
summary["acc"] = 100 * summary["acc"]
summary.head(20)  # acc em porcentagem por sequencia

In [ ]:
# =======================================
# Grafico 3: F1 score por trecho temporal (um unico grafico)
# =======================================
trecho_bins = [0.0, 0.25, 0.50, 0.75, 1.0]
trecho_labels = ["0-0.25", "0.25-0.5", "0.5-0.75", "0.75-1"]

preds_trecho = preds.copy()

# Garante rep_norm mesmo se a coluna ainda nao existir
if "rep_norm" not in preds_trecho.columns:
    preds_trecho = preds_trecho.sort_values(["fold", "video_id", "rep_id"]).reset_index(drop=True)
    preds_trecho["rep_norm"] = preds_trecho.groupby(["fold", "video_id"], group_keys=False)["rep_id"].apply(normalize_rep_id)

preds_trecho["trecho"] = pd.cut(
    preds_trecho["rep_norm"],
    bins=trecho_bins,
    labels=trecho_labels,
    include_lowest=True,
    right=True,
 )

def f1_binario_real(df_bin: pd.DataFrame) -> float:
    y_true = df_bin["y_true"].to_numpy()
    y_pred = df_bin["y_pred"].to_numpy()

    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()

    # Quando nao ha positivos reais nem preditos, F1 e indefinido para o trecho
    if ((y_true == 1).sum() == 0) and ((y_pred == 1).sum() == 0):
        return np.nan

    denom = 2 * tp + fp + fn
    return np.nan if denom == 0 else (2 * tp) / denom

# Distribuicao para o boxplot: F1 por fold em cada trecho
f1_por_fold_trecho = (
    preds_trecho.dropna(subset=["trecho"])
.groupby(["fold", "trecho"], observed=True)
.apply(f1_binario_real)
    .reset_index(name="f1")
)
f1_por_fold_trecho["trecho"] = f1_por_fold_trecho["trecho"].astype(str)

# Valor "em si": F1 global por trecho (todos os folds juntos)
f1_global_trecho = (
    preds_trecho.dropna(subset=["trecho"])
.groupby("trecho", observed=True)
.apply(f1_binario_real)
    .reindex(trecho_labels)
)

# Monta dados por posicao fixa para garantir exibicao dos 4 trechos
group_values = []
for lab in trecho_labels:
    vals = f1_por_fold_trecho.loc[f1_por_fold_trecho["trecho"] == lab, "f1"].dropna().to_numpy()
    group_values.append(vals if len(vals) > 0 else np.array([np.nan]))

fig3, ax3 = plt.subplots(figsize=(12, 5.8))
xpos = np.arange(len(trecho_labels))

bp = ax3.boxplot(
    group_values,
    positions=xpos,
    widths=0.58,
    patch_artist=True,
    showfliers=False,
    manage_ticks=False,
 )
for box in bp["boxes"]:
    box.set_facecolor("#bdd7e7")
    box.set_alpha(0.85)
for median in bp["medians"]:
    median.set_color("#4c4c4c")
    median.set_linewidth(1.2)

# Pontos por fold com jitter
rng = np.random.default_rng(2609)
for i, lab in enumerate(trecho_labels):
    vals = f1_por_fold_trecho.loc[f1_por_fold_trecho["trecho"] == lab, "f1"].dropna().to_numpy()
    if len(vals) > 0:
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax3.scatter(np.full(len(vals), i) + jitter, vals, s=20, color="black", alpha=0.55, zorder=3)

# Linha do F1 global por trecho
ax3.plot(
    xpos,
    f1_global_trecho.values,
    color="#d62728",
    marker="D",
    markersize=7,
    linewidth=1.8,
    label="F1 global por trecho",
)

for x, y in zip(xpos, f1_global_trecho.values):
    if pd.notna(y):
        ax3.text(x, min(1.0, y + 0.04), f"{y:.2f}", ha="center", va="bottom", fontsize=9, color="#d62728")
    else:
        ax3.text(x, 0.03, "indef.", ha="center", va="bottom", fontsize=9, color="#d62728")

ax3.set_title("F1 por trecho: distribuicao entre folds + F1 global")
ax3.set_xlabel("trecho do tempo normalizado")
ax3.set_ylabel("F1 score")
ax3.set_xticks(xpos)
ax3.set_xticklabels(trecho_labels)
ax3.set_ylim(-0.02, 1.02)
ax3.grid(True, axis="y", alpha=0.30)
ax3.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

f1_stats = (
    f1_por_fold_trecho.groupby("trecho", observed=True)["f1"]
    .agg(media_folds="mean", desvio_padrao_folds="std", n_folds_validos="count")
    .reindex(trecho_labels)
)
f1_stats["f1_global"] = f1_global_trecho
f1_stats

In [ ]:
# =======================================
# Plot 4: Confusion Matrix (TP, TN, FP, FN)
# =======================================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

if "preds" not in globals():
    pred_path = Path("saida/resultados/cv_predictions.csv")
    preds = pd.read_csv(pred_path)

y_true_cm = pd.to_numeric(preds["y_true"], errors="coerce").fillna(0).astype(int)
y_pred_cm = pd.to_numeric(preds["y_pred"], errors="coerce").fillna(0).astype(int)

tp = int(((y_true_cm == 1) & (y_pred_cm == 1)).sum())
tn = int(((y_true_cm == 0) & (y_pred_cm == 0)).sum())
fp = int(((y_true_cm == 0) & (y_pred_cm == 1)).sum())
fn = int(((y_true_cm == 1) & (y_pred_cm == 0)).sum())

cm = np.array([[tn, fp], [fn, tp]])

fig4, ax4 = plt.subplots(figsize=(8.6, 7.2))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    linewidths=1.2,
    linecolor="white",
    xticklabels=["Predicted 0", "Predicted 1"],
    yticklabels=["Actual 0", "Actual 1"],
    annot_kws={"fontsize": 25, "fontweight": "bold"},
    ax=ax4,
 )

#ax4.set_title("Confusion Matrix", fontsize=25, pad=14, weight="bold")
ax4.set_xlabel("Predicted Class", fontsize=20, labelpad=10, weight="bold")
ax4.set_ylabel("Actual Class", fontsize=20, labelpad=10, weight="bold")
ax4.tick_params(axis="x", labelsize=15)
ax4.tick_params(axis="y", labelsize=15)

# Semantic labels inside cells
# ax4.text(0.5, 0.15, "True Negative", ha="center", va="center", color="#ccd5df", fontsize=16, fontweight="bold")
# ax4.text(1.5, 0.15, "False Positive", ha="center", va="center", color="#1f4e79", fontsize=16, fontweight="bold")
# ax4.text(0.5, 1.15, "False Negative", ha="center", va="center", color="#1f4e79", fontsize=16, fontweight="bold")
# ax4.text(1.5, 1.15, "True Positive", ha="center", va="center", color="#1f4e79", fontsize=16, fontweight="bold")
ax4.text(0.5, 0.15, "True Negative", ha="center", va="center", color="#ccd5df", fontsize=16, fontweight="bold")
ax4.text(1.5, 0.15, "False Positive", ha="center", va="center", color="#1f4e79", fontsize=16, fontweight="bold")
ax4.text(0.5, 1.15, "False Negative", ha="center", va="center", color="#1f4e79", fontsize=16, fontweight="bold")
ax4.text(1.5, 1.15, "True Positive", ha="center", va="center", color="#1f4e79", fontsize=16, fontweight="bold")


plt.tight_layout()
plt.show()

pd.DataFrame(
    [{
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "True Positive": tp,
    }]
 )

In [ ]:
!pip install scikit-learn

In [ ]:
# =======================================
# Plot 5: ROC and PR-AUC Curves Side by Side
# =======================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

if "preds" not in globals():
    pred_path = Path("saida/resultados/cv_predictions.csv")
    preds = pd.read_csv(pred_path)

y_true_curve = pd.to_numeric(preds["y_true"], errors="coerce").fillna(0).astype(int)

# Prefer score/probability; if unavailable, fall back to y_pred (less informative)
score_candidates = [
    "y_score", "y_prob", "y_proba", "prob", "proba", "score",
    "pred_score", "pred_prob", "prob_1", "proba_1",
 ]
score_col = next((c for c in score_candidates if c in preds.columns), None)

if score_col is not None:
    y_score_curve = pd.to_numeric(preds[score_col], errors="coerce").fillna(0.0).astype(float)
    score_source = f"score: {score_col}"
else:
    y_score_curve = pd.to_numeric(preds["y_pred"], errors="coerce").fillna(0).astype(float)
    score_source = "fallback: y_pred (no probability)"

# ROC curve
fpr, tpr, _ = roc_curve(y_true_curve, y_score_curve)
roc_auc = auc(fpr, tpr)

# PR curve
precision, recall, _ = precision_recall_curve(y_true_curve, y_score_curve)
pr_auc = auc(recall, precision)
ap = average_precision_score(y_true_curve, y_score_curve)

fig5, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13.5, 5.5))

# ROC panel
ax_roc.plot(fpr, tpr, color="#1f77b4", linewidth=2.2, label=f"ROC AUC = {roc_auc:.4f}")
ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.3, label="baseline")
ax_roc.set_title("ROC Curve", fontsize=17, weight="bold")
ax_roc.set_xlabel("False Positive Rate", fontsize=15)
ax_roc.set_ylabel("True Positive Rate", fontsize=15)
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1.02)
ax_roc.grid(True, alpha=0.3)
ax_roc.legend(loc="lower right", frameon=True, fontsize=15)

# PR panel
base_pos = y_true_curve.mean()
ax_pr.plot(recall, precision, color="#d62728", linewidth=2.2, label=f"PR AUC = {pr_auc:.4f} | AP = {ap:.4f}")
ax_pr.hlines(base_pos, 0, 1, linestyle="--", color="gray", linewidth=1.3, label=f"baseline = {base_pos:.4f}")
ax_pr.set_title("Precision-Recall Curve", fontsize=17, weight="bold")
ax_pr.set_xlabel("Recall", fontsize=15)
ax_pr.set_ylabel("Precision", fontsize=15)
ax_pr.set_xlim(0, 1)
ax_pr.set_ylim(0, 1.02)
ax_pr.grid(True, alpha=0.3)
ax_pr.legend(loc="lower left", frameon=True, fontsize=15)

# fig5.suptitle(f"ROC and PR-AUC ({score_source})", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

pd.DataFrame(
    [{
        "score_source": score_source,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "average_precision": ap,
    }]
 )

## y_true x y_pred com rep_id normalizado

Este bloco usa as predições do modelo em `saida/resultados/cv_predictions.csv` (geradas no notebook 05-lstm) e normaliza `rep_id` em cada grupo `(fold, video_id)` para representar o progresso temporal da sequência no intervalo `[0, 1]`.

In [ ]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt

image_path = r"C:\Users\Gush\Downloads\20260102_182419.jpg"  # atualize com o caminho da sua imagem

image_bgr = cv2.imread(image_path)
if image_bgr is None:
    raise FileNotFoundError(f"Imagem nao encontrada: {image_path}")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

try:
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
except AttributeError:
    # fallback para instalacoes antigas/ambientes quebrados
    import mediapipe.python.solutions as mp_solutions
    mp_pose = mp_solutions.pose
    mp_drawing = mp_solutions.drawing_utils

pose = mp_pose.Pose(
    static_image_mode=True,
    model_complexity=2,
    enable_segmentation=False,
    min_detection_confidence=0.5,
)

results = pose.process(image_rgb)
annotated = image_rgb.copy()

if results.pose_landmarks:
    mp_drawing.draw_landmarks(
        annotated,
        results.pose_landmarks,
        mp_pose.POSE_CONNECTIONS,
    )

plt.figure(figsize=(6, 6))
plt.imshow(annotated)
plt.axis("off")

pose.close()